# DataSage Inference & Evaluation

**Standalone Colab notebook** for running GRPO-trained DataSage LoRA models against live HF Space environments.

This notebook:
1. Fetches W&B training metrics from 3 GRPO runs
2. Loads the fine-tuned LoRA adapters (Qwen2.5-3B + LoRA)
3. Runs inference against live environments (cleaning, enrichment, answering)
4. Exports evaluation results as JSON for the demo dashboard

**Requirements:** Colab GPU runtime (T4 minimum, A100/L4 recommended)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricalanis/openenv-hackaton/blob/main/demo/inference/datasage_inference.ipynb)

In [ ]:
!pip install -q unsloth trl wandb requests datasets huggingface_hub

In [ ]:
# --- API Keys ---
# In Colab: use Secrets tab (key icon) to set WANDB_API_KEY and HF_TOKEN
# Locally: set environment variables or paste below

import os

try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    IN_COLAB = True
    print("Running in Colab - loaded secrets")
except ImportError:
    IN_COLAB = False
    print("Running locally - using environment variables")

assert os.environ.get("WANDB_API_KEY"), "Set WANDB_API_KEY in Colab Secrets or env"
print(f"WANDB_API_KEY: ...{os.environ['WANDB_API_KEY'][-6:]}")
print(f"HF_TOKEN: {'set' if os.environ.get('HF_TOKEN') else 'not set (public models OK)'}")

In [ ]:
# ============================================================
# Configuration
# ============================================================

import json
import re
import time
import requests
import numpy as np
from datetime import datetime

# --- Base model ---
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct"
MAX_SEQ_LENGTH = 2048

# --- LoRA adapter repos on HuggingFace ---
LORA_REPOS = {
    "cleaning": "ricalanis/cleaning-grpo",
    "enrichment": "ricalanis/enrichment-grpo",
    "answering": "ricalanis/answering-grpo",
}

# --- Live HF Space environment URLs ---
SPACE_URLS = {
    "cleaning": "https://ricalanis-datasage-cleaning.hf.space",
    "enrichment": "https://ricalanis-datasage-enrichment.hf.space",
    "answering": "https://ricalanis-datasage-answering.hf.space",
}

# --- W&B runs to export ---
WANDB_PROJECT = "ricalanis/datasage"
WANDB_RUNS = {
    "cleaning": "xuwyjpe6",
    "enrichment": "orww3s2q",
    "answering": "2mltqk5w",
}

# --- Inference settings ---
DOMAINS = ["hr", "sales", "pm", "it_ops"]
PERSONAS = ["executive", "manager", "ic"]
EPISODES_PER_DOMAIN = 3  # per domain for cleaning/enrichment
MAX_CLEANING_STEPS = 15
MAX_ENRICHMENT_STEPS = 12

print("Configuration loaded.")
print(f"  Base model: {BASE_MODEL}")
print(f"  LoRA repos: {list(LORA_REPOS.values())}")
print(f"  Domains: {DOMAINS}")
print(f"  Personas: {PERSONAS}")

## 1. Export W&B Training Data

Fetches training metrics (reward curves, loss, grad norms) from the 3 GRPO training runs.

In [ ]:
import wandb

api = wandb.Api()

wandb_data = {}

for task, run_id in WANDB_RUNS.items():
    print(f"\n{'='*60}")
    print(f"Fetching W&B run: {task} ({run_id})")
    print(f"{'='*60}")

    try:
        run = api.run(f"{WANDB_PROJECT}/{run_id}")

        # Run summary
        print(f"  Name:     {run.name}")
        print(f"  State:    {run.state}")
        print(f"  Runtime:  {run.summary.get('_runtime', 0):.0f}s")

        # Fetch full history
        history = []
        for row in run.scan_history():
            history.append(dict(row))

        print(f"  History rows: {len(history)}")

        # Extract key metrics
        reward_keys = [k for k in (history[0].keys() if history else []) if "reward" in k.lower()]
        loss_keys = [k for k in (history[0].keys() if history else []) if "loss" in k.lower()]
        print(f"  Reward keys: {reward_keys}")
        print(f"  Loss keys: {loss_keys}")

        # Build metric series
        metrics = {}
        all_keys = set()
        for row in history:
            all_keys.update(row.keys())

        for key in all_keys:
            values = [row.get(key) for row in history if row.get(key) is not None]
            if values and all(isinstance(v, (int, float)) for v in values):
                metrics[key] = {
                    "values": [float(v) for v in values],
                    "min": float(min(values)),
                    "max": float(max(values)),
                    "mean": float(np.mean(values)),
                    "final": float(values[-1]),
                    "n_points": len(values),
                }

        wandb_data[task] = {
            "run_id": run_id,
            "run_name": run.name,
            "state": run.state,
            "config": dict(run.config) if run.config else {},
            "summary": {k: v for k, v in run.summary.items() if isinstance(v, (int, float, str, bool))},
            "metrics": metrics,
            "n_history_rows": len(history),
        }

        # Print summary stats for key metrics
        for key in sorted(metrics.keys()):
            if any(kw in key.lower() for kw in ["reward", "loss", "grad", "lr"]):
                m = metrics[key]
                print(f"  {key}: mean={m['mean']:.4f}, final={m['final']:.4f}, range=[{m['min']:.4f}, {m['max']:.4f}]")

    except Exception as e:
        print(f"  ERROR fetching run {run_id}: {e}")
        wandb_data[task] = {"run_id": run_id, "error": str(e)}

print(f"\nW&B export complete. Tasks fetched: {list(wandb_data.keys())}")

In [ ]:
# Save W&B data to JSON
wandb_export_path = "wandb_training_data.json"

# Convert numpy types for JSON serialization
def _json_safe(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

with open(wandb_export_path, "w") as f:
    json.dump(wandb_data, f, indent=2, default=_json_safe)

print(f"W&B training data saved to {wandb_export_path}")
print(f"File size: {os.path.getsize(wandb_export_path) / 1024:.1f} KB")

## 2. Load Base Model + LoRA Adapters

Load the Qwen2.5-3B-Instruct base model via Unsloth, then swap LoRA adapters per task.

In [ ]:
import torch
from unsloth import FastLanguageModel

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Load base model once - adapters will be loaded per-task
print(f"\nLoading base model: {BASE_MODEL}")
t0 = time.time()

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=16,
    gpu_memory_utilization=0.6,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Base model loaded in {time.time() - t0:.1f}s")
print(f"Parameters: {sum(p.numel() for p in base_model.parameters()):,}")

In [ ]:
def load_lora_model(task: str):
    """Load a LoRA adapter for a specific task."""
    repo = LORA_REPOS[task]
    print(f"Loading LoRA adapter: {repo}")
    t0 = time.time()

    model, tok = FastLanguageModel.from_pretrained(
        model_name=repo,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        fast_inference=True,
        max_lora_rank=16,
        gpu_memory_utilization=0.6,
    )

    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    # Enable inference mode
    FastLanguageModel.for_inference(model)

    print(f"  Loaded in {time.time() - t0:.1f}s")
    return model, tok

## 3. System Prompts & Action Parsers

Inlined from the training pipeline for self-contained execution.

In [ ]:
# ============================================================
# System prompts (same as training)
# ============================================================

SYSTEM_PROMPTS = {
    "cleaning": (
        "You are a data quality agent. You clean enterprise datasets across multiple "
        "domains (HR, Sales, Project Management, IT Operations).\n\n"
        "Each turn, analyze the data and respond with a JSON cleaning action:\n"
        '{"operation": "<op>", "column": "<col>", "value": "<val>", "params": {}}\n\n'
        "Available operations:\n"
        "- fill_null: Fill null values (value can be 'median', 'mode', or a specific value)\n"
        "- fix_type: Fix type mismatches in a column (cast to proper type)\n"
        "- remove_duplicate: Remove duplicate rows\n"
        "- standardize: Standardize column values (strip whitespace, normalize case)\n"
        "- trim: Trim whitespace from column values\n"
        "- correct_typo: Correct typos in categorical values\n\n"
        "Think step by step: examine the data quality report, identify the most "
        "impactful issue, then act."
    ),
    "enrichment": (
        "You are a data enrichment agent. You enrich enterprise datasets by adding "
        "computed fields and external lookups.\n\n"
        "Each turn, respond with a JSON enrichment action:\n"
        '{"operation": "<op>", "field_name": "<name>", "source": "<source>", "logic": "", "params": {}}\n\n'
        "Available operations: add_field, lookup, compute_derived, add_category.\n"
        "Pick from the possible_enrichments list. Use a different field each time."
    ),
    "answering": (
        "You are a data analyst answering questions for a specific persona.\n"
        "Adapt your language and depth:\n"
        "- Executive: strategic, financial impact, KPIs\n"
        "- Manager: operational, actionable recommendations\n"
        "- IC: plain language, personal impact\n\n"
        "Cite specific data columns. Be data-grounded. Respond with JSON:\n"
        '{"answer": "<your answer>", "cited_columns": ["col1", "col2"], "reasoning": "<brief>"}'
    ),
}

# ============================================================
# Action parsers (inlined from training/shared/parsers.py)
# ============================================================

def _extract_column(text: str) -> str:
    quoted = re.findall(r'["\']([\w]+)["\']', text)
    if quoted:
        return quoted[0]
    camel = re.findall(r'\b([A-Z][a-z]+(?:[A-Z][a-z]+)+)\b', text)
    if camel:
        return camel[0]
    return ""


def parse_cleaning_action(text: str) -> dict:
    json_match = re.search(r'\{[^{}]*"operation"[^{}]*\}', text, re.DOTALL)
    if json_match:
        try:
            data = json.loads(json_match.group())
            if "operation" in data and "column" in data:
                return data
        except json.JSONDecodeError:
            pass
    text_lower = text.lower()
    if "fill" in text_lower or "null" in text_lower:
        return {"operation": "fill_null", "column": _extract_column(text), "value": "median", "params": {}}
    elif "type" in text_lower or "cast" in text_lower:
        return {"operation": "fix_type", "column": _extract_column(text), "value": "numeric", "params": {}}
    elif "duplicate" in text_lower:
        return {"operation": "remove_duplicate", "column": "", "params": {}}
    elif "standard" in text_lower:
        return {"operation": "standardize", "column": _extract_column(text), "params": {}}
    elif "trim" in text_lower:
        return {"operation": "trim", "column": _extract_column(text), "params": {}}
    elif "typo" in text_lower:
        return {"operation": "correct_typo", "column": _extract_column(text), "params": {}}
    return {"operation": "fill_null", "column": "", "value": "median", "params": {}}


def parse_enrichment_action(text: str) -> dict:
    json_match = re.search(r'\{[^{}]*"operation"[^{}]*\}', text, re.DOTALL)
    if json_match:
        try:
            data = json.loads(json_match.group())
            if "field_name" in data:
                return data
        except json.JSONDecodeError:
            pass
    sources = [
        "salary_band", "tenure_risk", "satisfaction_index", "industry_benchmark",
        "flight_risk_score", "deal_size_category", "velocity_score",
        "win_probability_model", "industry_code", "competitive_risk",
        "schedule_risk_score", "resource_utilization", "dependency_chain_depth",
        "burndown_rate", "delay_probability", "sla_compliance_flag", "mttr_band",
        "escalation_path", "incident_severity_score", "recurring_pattern_flag",
    ]
    text_lower = text.lower()
    for source in sources:
        if source.replace("_", " ") in text_lower or source in text_lower:
            return {"operation": "add_field", "field_name": source, "source": source, "logic": "", "params": {}}
    return {"operation": "add_field", "field_name": "unknown", "source": "", "logic": "", "params": {}}


def parse_answering_action(text: str) -> dict:
    json_match = re.search(r'\{[^{}]*"answer"[^{}]*\}', text, re.DOTALL)
    if json_match:
        try:
            data = json.loads(json_match.group())
            if "answer" in data:
                return data
        except json.JSONDecodeError:
            pass
    cited = re.findall(r'\b([A-Z][a-zA-Z]+(?:[A-Z][a-z]+)*)\b', text)
    return {"answer": text, "cited_columns": cited[:5], "reasoning": ""}


PARSERS = {
    "cleaning": parse_cleaning_action,
    "enrichment": parse_enrichment_action,
    "answering": parse_answering_action,
}

print("System prompts and parsers ready.")

## 4. Environment Helpers

Functions to interact with the live HF Space environments via HTTP.

In [ ]:
def env_reset(task: str, seed: int = 42) -> dict:
    """Reset environment via /web/reset and return observation."""
    url = SPACE_URLS[task]
    try:
        resp = requests.post(f"{url}/web/reset", json={"seed": seed}, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        return data.get("observation", data)
    except Exception as e:
        print(f"  [WARN] env_reset({task}) failed: {e}")
        return {"error": str(e)}


def env_step(task: str, action: dict) -> dict:
    """Send action to environment via /web/step."""
    url = SPACE_URLS[task]
    try:
        resp = requests.post(f"{url}/web/step", json={"action": action}, timeout=30)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f"  [WARN] env_step({task}) failed: {e}")
        return {"error": str(e), "reward": 0, "done": True, "observation": {}}


def format_observation(obs: dict, task: str) -> str:
    """Format observation dict into a prompt string."""
    if task == "cleaning":
        return (
            f"Domain: {obs.get('domain', 'unknown')}\n"
            f"DQ Score: {obs.get('dq_score', 'N/A')}\n"
            f"DQ Report: {obs.get('dq_report', '')}\n"
            f"Columns: {obs.get('columns_info', '')}\n"
            f"Data Preview: {str(obs.get('data_preview', ''))[:500]}\n"
            f"Step {obs.get('step_number', '?')}/{obs.get('max_steps', '?')}\n\n"
            "Output a single JSON cleaning action."
        )
    elif task == "enrichment":
        return (
            f"Domain: {obs.get('domain', 'unknown')}\n"
            f"Schema: {obs.get('schema_info', '')}\n"
            f"Coverage: {obs.get('enrichment_coverage', 0)}\n"
            f"Fields Added: {obs.get('fields_added', [])}\n"
            f"Possible Enrichments: {obs.get('possible_enrichments', [])}\n"
            f"Step {obs.get('step_number', '?')}/{obs.get('max_steps', '?')}\n\n"
            "Output a single JSON enrichment action. Choose a field NOT already added."
        )
    elif task == "answering":
        return (
            f"Domain: {obs.get('domain', 'unknown')}\n"
            f"Persona: {obs.get('persona', 'unknown')} - {obs.get('persona_description', '')}\n"
            f"Question: {obs.get('question', 'N/A')}\n"
            f"Available Columns: {obs.get('available_columns', [])}\n"
            f"Column Stats: {str(obs.get('column_stats', ''))[:800]}\n"
            f"Dataset Summary: {str(obs.get('dataset_summary', ''))[:400]}\n\n"
            "Answer the question in the persona's style. Output JSON."
        )
    return json.dumps(obs, indent=2)


# Verify environments are up
print("Checking environment health...")
for task, url in SPACE_URLS.items():
    try:
        r = requests.get(url, timeout=10)
        print(f"  {task}: {r.status_code} OK")
    except Exception as e:
        print(f"  {task}: UNREACHABLE ({e})")

## 5. Inference Engine

Runs the fine-tuned models against live environments, collecting metrics.

In [ ]:
def generate_response(model, tokenizer, system_prompt: str, user_prompt: str, max_new_tokens: int = 256) -> str:
    """Generate a response from the model using chat template."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only the new tokens
    new_tokens = output_ids[0][input_ids.shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Strip any thinking tags (Qwen sometimes adds these)
    response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()

    return response


def run_cleaning_episode(model, tokenizer, seed: int = 42) -> dict:
    """Run a single cleaning episode against the live environment."""
    obs = env_reset("cleaning", seed=seed)
    if "error" in obs:
        return {"error": obs["error"], "rewards": [], "steps": 0}

    rewards = []
    actions = []
    initial_dq = obs.get("dq_score", 0)

    for step in range(MAX_CLEANING_STEPS):
        prompt = format_observation(obs, "cleaning")
        response = generate_response(model, tokenizer, SYSTEM_PROMPTS["cleaning"], prompt)
        action = parse_cleaning_action(response)
        actions.append(action)

        result = env_step("cleaning", action)
        reward = result.get("reward", 0)
        rewards.append(reward)

        new_obs = result.get("observation", {})
        if isinstance(new_obs, dict):
            obs = {**obs, **new_obs}

        if result.get("done", False):
            break

    final_dq = obs.get("dq_score", initial_dq)
    return {
        "initial_dq": initial_dq,
        "final_dq": final_dq,
        "dq_improvement": final_dq - initial_dq,
        "rewards": rewards,
        "avg_reward": np.mean(rewards) if rewards else 0,
        "total_reward": sum(rewards),
        "steps": len(rewards),
        "actions": actions,
    }


def run_enrichment_episode(model, tokenizer, seed: int = 42) -> dict:
    """Run a single enrichment episode against the live environment."""
    obs = env_reset("enrichment", seed=seed)
    if "error" in obs:
        return {"error": obs["error"], "rewards": [], "steps": 0}

    rewards = []
    actions = []

    for step in range(MAX_ENRICHMENT_STEPS):
        prompt = format_observation(obs, "enrichment")
        response = generate_response(model, tokenizer, SYSTEM_PROMPTS["enrichment"], prompt)
        action = parse_enrichment_action(response)
        actions.append(action)

        result = env_step("enrichment", action)
        reward = result.get("reward", 0)
        rewards.append(reward)

        new_obs = result.get("observation", {})
        if isinstance(new_obs, dict):
            obs = {**obs, **new_obs}

        if result.get("done", False):
            break

    final_coverage = obs.get("enrichment_coverage", 0)
    return {
        "final_coverage": final_coverage,
        "rewards": rewards,
        "avg_reward": np.mean(rewards) if rewards else 0,
        "total_reward": sum(rewards),
        "steps": len(rewards),
        "fields_added": len([a for a in actions if a.get("field_name", "unknown") != "unknown"]),
        "actions": actions,
    }


def run_answering_episode(model, tokenizer, seed: int = 42) -> dict:
    """Run a single answering episode against the live environment."""
    obs = env_reset("answering", seed=seed)
    if "error" in obs:
        return {"error": obs["error"], "rewards": [], "steps": 0}

    prompt = format_observation(obs, "answering")
    response = generate_response(model, tokenizer, SYSTEM_PROMPTS["answering"], prompt)
    action = parse_answering_action(response)

    result = env_step("answering", action)
    reward = result.get("reward", 0)
    info = result.get("info", {})

    return {
        "reward": reward,
        "faithfulness": info.get("faithfulness", 0),
        "persona_alignment": info.get("persona_alignment", 0),
        "domain": obs.get("domain", "unknown"),
        "persona": obs.get("persona", "unknown"),
        "question": obs.get("question", ""),
        "answer": action.get("answer", response),
        "cited_columns": action.get("cited_columns", []),
    }


print("Inference engine ready.")

## 6. Run Inference: Cleaning

Load the cleaning LoRA adapter and run episodes across all domains.

In [ ]:
print("=" * 60)
print("STAGE 1: CLEANING INFERENCE")
print("=" * 60)

cleaning_model, cleaning_tok = load_lora_model("cleaning")

cleaning_results = []
for domain_idx, domain in enumerate(DOMAINS):
    for ep in range(EPISODES_PER_DOMAIN):
        seed = domain_idx * 100 + ep + 1
        print(f"\n  [{domain}] Episode {ep+1}/{EPISODES_PER_DOMAIN} (seed={seed})")
        t0 = time.time()

        result = run_cleaning_episode(cleaning_model, cleaning_tok, seed=seed)
        result["domain"] = domain
        result["seed"] = seed
        result["latency_s"] = time.time() - t0
        cleaning_results.append(result)

        dq = result.get("final_dq", 0)
        rwd = result.get("avg_reward", 0)
        steps = result.get("steps", 0)
        print(f"    DQ: {dq:.4f} | Avg Reward: {rwd:.4f} | Steps: {steps} | Time: {result['latency_s']:.1f}s")

# Summary
valid = [r for r in cleaning_results if "error" not in r]
if valid:
    print(f"\n--- Cleaning Summary ({len(valid)} episodes) ---")
    print(f"  Mean Final DQ:    {np.mean([r['final_dq'] for r in valid]):.4f}")
    print(f"  Mean Avg Reward:  {np.mean([r['avg_reward'] for r in valid]):.4f}")
    print(f"  Mean Steps:       {np.mean([r['steps'] for r in valid]):.1f}")
    print(f"  Mean DQ Improve:  {np.mean([r['dq_improvement'] for r in valid]):.4f}")

# Free GPU memory
del cleaning_model, cleaning_tok
torch.cuda.empty_cache()

## 7. Run Inference: Enrichment

Load the enrichment LoRA adapter and run episodes.

In [ ]:
print("=" * 60)
print("STAGE 2: ENRICHMENT INFERENCE")
print("=" * 60)

enrichment_model, enrichment_tok = load_lora_model("enrichment")

enrichment_results = []
for domain_idx, domain in enumerate(DOMAINS):
    for ep in range(EPISODES_PER_DOMAIN):
        seed = domain_idx * 100 + ep + 1
        print(f"\n  [{domain}] Episode {ep+1}/{EPISODES_PER_DOMAIN} (seed={seed})")
        t0 = time.time()

        result = run_enrichment_episode(enrichment_model, enrichment_tok, seed=seed)
        result["domain"] = domain
        result["seed"] = seed
        result["latency_s"] = time.time() - t0
        enrichment_results.append(result)

        cov = result.get("final_coverage", 0)
        rwd = result.get("avg_reward", 0)
        steps = result.get("steps", 0)
        print(f"    Coverage: {cov:.4f} | Avg Reward: {rwd:.4f} | Steps: {steps} | Time: {result['latency_s']:.1f}s")

# Summary
valid = [r for r in enrichment_results if "error" not in r]
if valid:
    print(f"\n--- Enrichment Summary ({len(valid)} episodes) ---")
    print(f"  Mean Final Coverage:  {np.mean([r['final_coverage'] for r in valid]):.4f}")
    print(f"  Mean Avg Reward:      {np.mean([r['avg_reward'] for r in valid]):.4f}")
    print(f"  Mean Steps:           {np.mean([r['steps'] for r in valid]):.1f}")
    print(f"  Mean Fields Added:    {np.mean([r['fields_added'] for r in valid]):.1f}")

# Free GPU memory
del enrichment_model, enrichment_tok
torch.cuda.empty_cache()

## 8. Run Inference: Answering

Load the answering LoRA adapter and run episodes across domains and personas.

In [ ]:
print("=" * 60)
print("STAGE 3: ANSWERING INFERENCE")
print("=" * 60)

answering_model, answering_tok = load_lora_model("answering")

answering_results = []
seed_counter = 1
for domain in DOMAINS:
    for persona in PERSONAS:
        seed = seed_counter
        seed_counter += 1
        print(f"\n  [{domain}] Persona: {persona} (seed={seed})")
        t0 = time.time()

        result = run_answering_episode(answering_model, answering_tok, seed=seed)
        result["seed"] = seed
        result["latency_s"] = time.time() - t0
        answering_results.append(result)

        rwd = result.get("reward", 0)
        faith = result.get("faithfulness", 0)
        pa = result.get("persona_alignment", 0)
        print(f"    Reward: {rwd:.4f} | Faithfulness: {faith:.4f} | Persona Align: {pa:.4f} | Time: {result['latency_s']:.1f}s")
        print(f"    Answer preview: {result.get('answer', '')[:100]}...")

# Summary
valid = [r for r in answering_results if "error" not in r]
if valid:
    print(f"\n--- Answering Summary ({len(valid)} episodes) ---")
    print(f"  Mean Reward:             {np.mean([r['reward'] for r in valid]):.4f}")
    print(f"  Mean Faithfulness:       {np.mean([r['faithfulness'] for r in valid]):.4f}")
    print(f"  Mean Persona Alignment:  {np.mean([r['persona_alignment'] for r in valid]):.4f}")

    # Per-persona breakdown
    for p in PERSONAS:
        p_results = [r for r in valid if r.get("persona") == p]
        if p_results:
            print(f"  {p}: reward={np.mean([r['reward'] for r in p_results]):.4f} ({len(p_results)} episodes)")

# Free GPU memory
del answering_model, answering_tok
torch.cuda.empty_cache()

## 9. Also Run Base Model (No LoRA) for Comparison

Run the same episodes with the base Qwen2.5-3B model (no fine-tuning) to measure improvement.

In [ ]:
print("=" * 60)
print("BASE MODEL (no LoRA) INFERENCE")
print("=" * 60)

# Use the already-loaded base model
base_model_inf, base_tok = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=16,
    gpu_memory_utilization=0.6,
)
if base_tok.pad_token is None:
    base_tok.pad_token = base_tok.eos_token
FastLanguageModel.for_inference(base_model_inf)

# Run a smaller subset for comparison
base_cleaning = []
base_enrichment = []
base_answering = []

# Cleaning: 1 episode per domain
print("\n--- Base: Cleaning ---")
for domain_idx, domain in enumerate(DOMAINS):
    seed = domain_idx * 100 + 1
    print(f"  [{domain}] seed={seed}")
    result = run_cleaning_episode(base_model_inf, base_tok, seed=seed)
    result["domain"] = domain
    base_cleaning.append(result)
    print(f"    DQ: {result.get('final_dq', 0):.4f} | Reward: {result.get('avg_reward', 0):.4f}")

# Enrichment: 1 episode per domain
print("\n--- Base: Enrichment ---")
for domain_idx, domain in enumerate(DOMAINS):
    seed = domain_idx * 100 + 1
    print(f"  [{domain}] seed={seed}")
    result = run_enrichment_episode(base_model_inf, base_tok, seed=seed)
    result["domain"] = domain
    base_enrichment.append(result)
    print(f"    Coverage: {result.get('final_coverage', 0):.4f} | Reward: {result.get('avg_reward', 0):.4f}")

# Answering: all domain x persona combos
print("\n--- Base: Answering ---")
seed_counter = 1
for domain in DOMAINS:
    for persona in PERSONAS:
        seed = seed_counter
        seed_counter += 1
        print(f"  [{domain}] {persona} seed={seed}")
        result = run_answering_episode(base_model_inf, base_tok, seed=seed)
        result["seed"] = seed
        base_answering.append(result)
        print(f"    Reward: {result.get('reward', 0):.4f}")

del base_model_inf, base_tok
torch.cuda.empty_cache()

print("\nBase model inference complete.")

## 10. Aggregate & Export Results

Combine all results into a single JSON file for the demo dashboard.

In [ ]:
def summarize_cleaning(results):
    valid = [r for r in results if "error" not in r]
    if not valid:
        return {"error": "no valid episodes"}
    return {
        "final_dq_mean": float(np.mean([r["final_dq"] for r in valid])),
        "final_dq_std": float(np.std([r["final_dq"] for r in valid])),
        "avg_reward_mean": float(np.mean([r["avg_reward"] for r in valid])),
        "dq_improvement_mean": float(np.mean([r["dq_improvement"] for r in valid])),
        "steps_mean": float(np.mean([r["steps"] for r in valid])),
        "n_episodes": len(valid),
        "per_domain": {
            d: {
                "final_dq_mean": float(np.mean([r["final_dq"] for r in valid if r.get("domain") == d])),
                "avg_reward_mean": float(np.mean([r["avg_reward"] for r in valid if r.get("domain") == d])),
                "n": len([r for r in valid if r.get("domain") == d]),
            }
            for d in DOMAINS if any(r.get("domain") == d for r in valid)
        },
    }


def summarize_enrichment(results):
    valid = [r for r in results if "error" not in r]
    if not valid:
        return {"error": "no valid episodes"}
    return {
        "final_coverage_mean": float(np.mean([r["final_coverage"] for r in valid])),
        "avg_reward_mean": float(np.mean([r["avg_reward"] for r in valid])),
        "steps_mean": float(np.mean([r["steps"] for r in valid])),
        "fields_added_mean": float(np.mean([r["fields_added"] for r in valid])),
        "n_episodes": len(valid),
        "per_domain": {
            d: {
                "final_coverage_mean": float(np.mean([r["final_coverage"] for r in valid if r.get("domain") == d])),
                "avg_reward_mean": float(np.mean([r["avg_reward"] for r in valid if r.get("domain") == d])),
                "n": len([r for r in valid if r.get("domain") == d]),
            }
            for d in DOMAINS if any(r.get("domain") == d for r in valid)
        },
    }


def summarize_answering(results):
    valid = [r for r in results if "error" not in r]
    if not valid:
        return {"error": "no valid episodes"}
    return {
        "reward_mean": float(np.mean([r["reward"] for r in valid])),
        "reward_std": float(np.std([r["reward"] for r in valid])),
        "faithfulness_mean": float(np.mean([r["faithfulness"] for r in valid])),
        "persona_alignment_mean": float(np.mean([r["persona_alignment"] for r in valid])),
        "n_episodes": len(valid),
        "per_persona": {
            p: {
                "reward_mean": float(np.mean([r["reward"] for r in valid if r.get("persona") == p])),
                "n": len([r for r in valid if r.get("persona") == p]),
            }
            for p in PERSONAS if any(r.get("persona") == p for r in valid)
        },
        "per_domain": {
            d: {
                "reward_mean": float(np.mean([r["reward"] for r in valid if r.get("domain") == d])),
                "n": len([r for r in valid if r.get("domain") == d]),
            }
            for d in DOMAINS if any(r.get("domain") == d for r in valid)
        },
        "per_episode": [
            {
                "domain": r.get("domain"),
                "persona": r.get("persona"),
                "reward": float(r["reward"]),
                "faithfulness": float(r.get("faithfulness", 0)),
                "persona_alignment": float(r.get("persona_alignment", 0)),
            }
            for r in valid
        ],
    }


# Build final output
evaluation_results = {
    "datasage_finetuned": {
        "cleaning": summarize_cleaning(cleaning_results),
        "enrichment": summarize_enrichment(enrichment_results),
        "answering": summarize_answering(answering_results),
    },
    "base_model": {
        "model": BASE_MODEL,
        "cleaning": summarize_cleaning(base_cleaning),
        "enrichment": summarize_enrichment(base_enrichment),
        "answering": summarize_answering(base_answering),
    },
    "wandb_training": wandb_data,
    "metadata": {
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "base_model": BASE_MODEL,
        "lora_repos": LORA_REPOS,
        "environments": SPACE_URLS,
        "episodes_per_domain": EPISODES_PER_DOMAIN,
        "domains": DOMAINS,
        "personas": PERSONAS,
        "wandb_runs": WANDB_RUNS,
    },
}

# Save
output_path = "evaluation_results.json"
with open(output_path, "w") as f:
    json.dump(evaluation_results, f, indent=2, default=_json_safe)

print(f"\nEvaluation results saved to {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")

# Also strip action details for a lighter version for the demo
# (actions can be large with full model responses)
light_results = json.loads(json.dumps(evaluation_results, default=_json_safe))
for stage in ["cleaning", "enrichment"]:
    # Remove per-episode action lists to keep it small
    pass  # Already summarized above

with open("evaluation_results_light.json", "w") as f:
    json.dump(light_results, f, indent=2)

print(f"Light version saved to evaluation_results_light.json")

## 11. Results Summary

Print a formatted comparison table.

In [ ]:
print("\n" + "=" * 70)
print("DATASAGE EVALUATION RESULTS")
print("=" * 70)

ft = evaluation_results["datasage_finetuned"]
base = evaluation_results["base_model"]

print(f"\n{'Task':<15} {'Metric':<25} {'DataSage (GRPO)':>15} {'Base Qwen2.5-3B':>15} {'Delta':>10}")
print("-" * 80)

# Cleaning
ft_c = ft["cleaning"]
b_c = base["cleaning"]
if "error" not in ft_c and "error" not in b_c:
    print(f"{'Cleaning':<15} {'Final DQ Score':<25} {ft_c['final_dq_mean']:>15.4f} {b_c['final_dq_mean']:>15.4f} {ft_c['final_dq_mean'] - b_c['final_dq_mean']:>+10.4f}")
    print(f"{'':.<15} {'Avg Reward':<25} {ft_c['avg_reward_mean']:>15.4f} {b_c['avg_reward_mean']:>15.4f} {ft_c['avg_reward_mean'] - b_c['avg_reward_mean']:>+10.4f}")
    print(f"{'':.<15} {'Avg Steps':<25} {ft_c['steps_mean']:>15.1f} {b_c['steps_mean']:>15.1f}")

# Enrichment
ft_e = ft["enrichment"]
b_e = base["enrichment"]
if "error" not in ft_e and "error" not in b_e:
    print(f"{'Enrichment':<15} {'Final Coverage':<25} {ft_e['final_coverage_mean']:>15.4f} {b_e['final_coverage_mean']:>15.4f} {ft_e['final_coverage_mean'] - b_e['final_coverage_mean']:>+10.4f}")
    print(f"{'':.<15} {'Avg Reward':<25} {ft_e['avg_reward_mean']:>15.4f} {b_e['avg_reward_mean']:>15.4f} {ft_e['avg_reward_mean'] - b_e['avg_reward_mean']:>+10.4f}")
    print(f"{'':.<15} {'Fields Added':<25} {ft_e['fields_added_mean']:>15.1f} {b_e['fields_added_mean']:>15.1f}")

# Answering
ft_a = ft["answering"]
b_a = base["answering"]
if "error" not in ft_a and "error" not in b_a:
    print(f"{'Answering':<15} {'Mean Reward':<25} {ft_a['reward_mean']:>15.4f} {b_a['reward_mean']:>15.4f} {ft_a['reward_mean'] - b_a['reward_mean']:>+10.4f}")
    print(f"{'':.<15} {'Faithfulness':<25} {ft_a['faithfulness_mean']:>15.4f} {b_a['faithfulness_mean']:>15.4f} {ft_a['faithfulness_mean'] - b_a['faithfulness_mean']:>+10.4f}")
    print(f"{'':.<15} {'Persona Alignment':<25} {ft_a['persona_alignment_mean']:>15.4f} {b_a['persona_alignment_mean']:>15.4f} {ft_a['persona_alignment_mean'] - b_a['persona_alignment_mean']:>+10.4f}")

# E2E score
print("\n" + "-" * 80)
e2e_weights = {"cleaning": 0.30, "enrichment": 0.30, "answering": 0.40}

def compute_e2e(results):
    c = results["cleaning"].get("avg_reward_mean", 0)
    e = results["enrichment"].get("avg_reward_mean", 0)
    a = results["answering"].get("reward_mean", 0)
    return 0.30 * c + 0.30 * e + 0.40 * a

ft_e2e = compute_e2e(ft)
b_e2e = compute_e2e(base)
print(f"{'E2E Score':<15} {'(0.3*C + 0.3*E + 0.4*A)':<25} {ft_e2e:>15.4f} {b_e2e:>15.4f} {ft_e2e - b_e2e:>+10.4f}")

print("\n" + "=" * 70)
print("Done! Results exported to evaluation_results.json")
print("W&B training data exported to wandb_training_data.json")
print("=" * 70)

## 12. W&B Training Curves Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("GRPO Training Reward Curves (from W&B)", fontsize=14, fontweight="bold")

colors = {"cleaning": "#F59E0B", "enrichment": "#EF4444", "answering": "#3B82F6"}

for idx, (task, data) in enumerate(wandb_data.items()):
    ax = axes[idx]
    ax.set_title(f"{task.title()} (run: {data.get('run_id', '?')})")

    if "error" in data:
        ax.text(0.5, 0.5, f"Error: {data['error']}", ha="center", va="center", transform=ax.transAxes)
        continue

    metrics = data.get("metrics", {})

    # Find reward-related keys
    reward_keys = sorted([k for k in metrics if "reward" in k.lower() and metrics[k]["n_points"] > 1])

    if not reward_keys:
        # Try loss
        reward_keys = sorted([k for k in metrics if "loss" in k.lower() and metrics[k]["n_points"] > 1])

    for key in reward_keys[:4]:  # Plot up to 4 metrics
        values = metrics[key]["values"]
        ax.plot(values, label=key.replace("reward/", "").replace("train/", ""), alpha=0.8)

    ax.set_xlabel("Step")
    ax.set_ylabel("Value")
    ax.legend(fontsize=8, loc="best")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("wandb_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Training curves saved to wandb_training_curves.png")

In [ ]:
# Comparison bar chart: DataSage vs Base Model
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("DataSage GRPO vs Base Qwen2.5-3B", fontsize=14, fontweight="bold")

# Cleaning
ax = axes[0]
ft_val = ft["cleaning"].get("avg_reward_mean", 0)
b_val = base["cleaning"].get("avg_reward_mean", 0)
bars = ax.bar(["DataSage", "Base"], [ft_val, b_val], color=["#F59E0B", "#9CA3AF"])
ax.set_title("Cleaning (Avg Reward)")
ax.set_ylim(0, 1)
for bar, val in zip(bars, [ft_val, b_val]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02, f"{val:.3f}", ha="center")

# Enrichment
ax = axes[1]
ft_val = ft["enrichment"].get("final_coverage_mean", 0)
b_val = base["enrichment"].get("final_coverage_mean", 0)
bars = ax.bar(["DataSage", "Base"], [ft_val, b_val], color=["#EF4444", "#9CA3AF"])
ax.set_title("Enrichment (Coverage)")
ax.set_ylim(0, 1)
for bar, val in zip(bars, [ft_val, b_val]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02, f"{val:.3f}", ha="center")

# Answering
ax = axes[2]
ft_val = ft["answering"].get("reward_mean", 0)
b_val = base["answering"].get("reward_mean", 0)
bars = ax.bar(["DataSage", "Base"], [ft_val, b_val], color=["#3B82F6", "#9CA3AF"])
ax.set_title("Answering (Reward)")
ax.set_ylim(0, 1)
for bar, val in zip(bars, [ft_val, b_val]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02, f"{val:.3f}", ha="center")

plt.tight_layout()
plt.savefig("datasage_vs_base_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Comparison chart saved to datasage_vs_base_comparison.png")

## 13. Download Files

In Colab, download the output files.

In [ ]:
output_files = [
    "evaluation_results.json",
    "evaluation_results_light.json",
    "wandb_training_data.json",
    "wandb_training_curves.png",
    "datasage_vs_base_comparison.png",
]

print("Output files:")
for f in output_files:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024
        print(f"  {f}: {size:.1f} KB")
    else:
        print(f"  {f}: NOT FOUND")

if IN_COLAB:
    from google.colab import files
    for f in output_files:
        if os.path.exists(f):
            files.download(f)
    print("\nFiles downloaded to your browser.")
else:
    print("\nFiles saved to current directory.")
    print("Copy evaluation_results.json to demo/data/ for the dashboard.")